## Leave-One-Protocol-Out（LOPO）

跨协议检测常用设定：

1. **训练**：清零某一协议通道特征（模型从未见过该协议）
2. **测试**：官方 TEST **全特征**（该协议在测试时首次出现）
3. 主表：**Target Macro-F1**（8 折 mean±std）、**Worst**、**Gap = Source − Target**

对比 Ours（软融合）与 CKAN / GRID / WaveMamba / Transformer-IDS / FeCo / MPGNN / TCG-IDS。

结果文件：`lopo_leave_one_protocol_out.csv`（由 `_run_lopo.py` 生成）。


In [ ]:
# ===== LOPO：Target 指标 mean±std（8 折，格式同 multiseed） =====
import os
import numpy as np
import pandas as pd

_ROOT = os.path.abspath(os.getcwd())
for _cand in [_ROOT, os.path.dirname(_ROOT), r'E:\apt\YES\1']:
    if os.path.isfile(os.path.join(_cand, 'lopo_leave_one_protocol_out.csv')):
        _ROOT = _cand
        os.chdir(_ROOT)
        break

csv_p = os.path.join(_ROOT, 'lopo_leave_one_protocol_out.csv')
metrics_p = os.path.join(_ROOT, 'lopo_leave_one_protocol_metrics.csv')
assert os.path.isfile(csv_p), (
    f'缺少 {csv_p}\n请先运行: python _run_lopo.py  （约数小时，含 8 协议 × 9 模型重训）')

df_main = pd.read_csv(csv_p, index_col=0)
proto_cols = [c for c in df_main.columns if c.startswith('T_')]

METRICS = ['Accuracy', 'Precision', 'Recall', 'F1']


def fmt_pm(mean, std, digits=4):
    return f'{mean:.{digits}f}±{std:.{digits}f}'


def build_metrics_table():
    if os.path.isfile(metrics_p):
        dm = pd.read_csv(metrics_p, index_col=0)
        return dm.sort_values('F1_mean', ascending=False)

    rows = []
    for model, row in df_main.iterrows():
        vals = row[proto_cols].astype(float).values
        rec = {
            'F1_mean': float(vals.mean()),
            'F1_std': float(vals.std(ddof=1)),
        }
        if 'Target_mean' in row and pd.notna(row['Target_mean']):
            rec['F1_mean'] = float(row['Target_mean'])
            rec['F1_std'] = float(row['Target_std'])
        rows.append({'model': model, **rec})
    return pd.DataFrame(rows).set_index('model').sort_values('F1_mean', ascending=False)


dm = build_metrics_table()
has_full = all(f'{m}_mean' in dm.columns and f'{m}_std' in dm.columns for m in METRICS)

print('LOPO Target metrics (8-fold mean±std; train w/o held-out protocol → test full features)\n')
hdr = f'{"Model":18s}  ' + '  '.join(f'{m:>16s}' for m in METRICS)
print(hdr)
print('-' * len(hdr))
for model, row in dm.iterrows():
    bits = []
    for m in METRICS:
        mc, sc = f'{m}_mean', f'{m}_std'
        if mc in row.index and sc in row.index and pd.notna(row[mc]):
            bits.append(f'{fmt_pm(row[mc], row[sc]):>16s}')
        else:
            bits.append(f'{"—":>16s}')
    print(f'{model:18s}  ' + '  '.join(bits))

if not has_full:
    print('\n[提示] 当前仅有 F1（来自 lopo_leave_one_protocol_out.csv）。'
          '重跑 python _run_lopo.py 后会生成 lopo_leave_one_protocol_metrics.csv，'
          '届时 Accuracy / Precision / Recall 也会显示。')

print('\n' + '=' * 72)
print('Per-model detail (LOPO Target, 8-fold mean±std)')
print('=' * 72)
for model, row in dm.iterrows():
    print(f'\n{model}')
    for m in METRICS:
        mc, sc = f'{m}_mean', f'{m}_std'
        if mc in row.index and sc in row.index and pd.notna(row[mc]):
            print(f'  {m:10s}: {fmt_pm(row[mc], row[sc])}')

# Excel 可粘贴：制表符分隔
print('\n' + '=' * 72)
print('Excel paste (tab-separated)')
print('=' * 72)
print('Model\t' + '\t'.join(METRICS))
for model, row in dm.iterrows():
    vals = []
    for m in METRICS:
        mc, sc = f'{m}_mean', f'{m}_std'
        vals.append(fmt_pm(row[mc], row[sc]) if mc in row.index and pd.notna(row.get(mc)) else '—')
    print(f'{model}\t' + '\t'.join(vals))

# 辅助：F1 汇总 / 逐协议
cols = [c for c in ['Source_F1', 'Target_mean', 'Target_std', 'Target_worst', 'Gap_mean', 'Gap_worst']
        if c in df_main.columns]
show = df_main.sort_values('Target_mean', ascending=False)
print('\nLOPO F1 summary')
print(show[cols].round(4).to_string())
print('\nPer-protocol Target F1:')
print(show[proto_cols].round(4).to_string())


## Leave-k-Protocols-Out（k=2 / 3 / 4）

在 LOPO（留 1 协议）基础上的扩展：

1. **训练/验证**：清零 **k 个协议**通道（遍历全部组合：k=2 共 28，k=3 共 56，k=4 共 70）
2. **测试**：官方 TEST **全协议特征**
3. 主表：Accuracy / Precision / Recall / F1 的 **mean±std**

运行：`python _run_lopo_k.py`（当前脚本跑 k=3、k=4；k=2 已完成会并入总表）

缓存：`_tmp/lopo_cache/lopo_k_cache/`  
结果：`lopo_leave{k}_protocol_metrics.csv`、`lopo_leave_k_protocol_metrics.csv`

In [ ]:
# ===== Leave-k-Protocols-Out：四项指标 mean±std =====
import os
import pandas as pd

_ROOT = os.path.abspath(os.getcwd())
for _cand in [_ROOT, os.path.dirname(_ROOT), '/root/YES/YES/1', r'E:\apt\YES\1']:
    if os.path.isfile(os.path.join(_cand, 'lopo_leave2_protocol_metrics.csv')):
        _ROOT = _cand
        os.chdir(_ROOT)
        break

METRICS = ['Accuracy', 'Precision', 'Recall', 'F1']
K_VALUES = [2, 3, 4]


def fmt_pm(mean, std, digits=4):
    return f'{mean:.{digits}f}±{std:.{digits}f}'


def print_k_table(k):
    metrics_p = os.path.join(_ROOT, f'lopo_leave{k}_protocol_metrics.csv')
    if not os.path.isfile(metrics_p):
        print(f'[k={k}] 缺少 {metrics_p}，请先运行: python _run_lopo_k.py')
        return
    dm = pd.read_csv(metrics_p, index_col=0).sort_values('F1_mean', ascending=False)
    n_folds = int(dm['n_folds'].iloc[0]) if 'n_folds' in dm.columns else '—'
    print(f'\nLeave-{k}-Protocols-Out Target metrics ({n_folds}-combo mean±std; train w/o k protocols → test full)\n')
    hdr = f'{"Model":18s}  ' + '  '.join(f'{m:>16s}' for m in METRICS)
    print(hdr)
    print('-' * len(hdr))
    for model, row in dm.iterrows():
        bits = []
        for m in METRICS:
            mc, sc = f'{m}_mean', f'{m}_std'
            bits.append(f'{fmt_pm(row[mc], row[sc]):>16s}' if pd.notna(row.get(mc)) else f'{"—":>16s}')
        print(f'{model:18s}  ' + '  '.join(bits))

    print('\nExcel paste (tab-separated)')
    print('Model\t' + '\t'.join(METRICS))
    for model, row in dm.iterrows():
        vals = [fmt_pm(row[f'{m}_mean'], row[f'{m}_std']) for m in METRICS]
        print(f'{model}\t' + '\t'.join(vals))


for k in K_VALUES:
    print_k_table(k)

combined_p = os.path.join(_ROOT, 'lopo_leave_k_protocol_metrics.csv')
if os.path.isfile(combined_p):
    df = pd.read_csv(combined_p)
    print('\n' + '=' * 72)
    print('Combined:', combined_p)
    cols = [c for c in ['k', 'model', 'Accuracy_mean', 'F1_mean', 'F1_std', 'n_folds'] if c in df.columns]
    print(df[cols].round(4).to_string(index=False))
